[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uwarring82/iontrap-dynamics/blob/main/docs/tutorials/notebooks/19_squeezing_by_quenching.ipynb)

**Run the _Setup_ cell below first**, then run the remaining cells top-to-bottom. The `assert` statements are the tutorial's built-in checks — if every cell runs without error, every step passed.

> _Auto-generated from [`docs/tutorials/19_squeezing_by_quenching.md`](https://github.com/uwarring82/iontrap-dynamics/blob/main/docs/tutorials/19_squeezing_by_quenching.md) by `tools/build_tutorial_notebooks.py`. Edit the Markdown tutorial, not this notebook._

In [ ]:
# Setup — install iontrap-dynamics and its dependencies (qutip, numpy, scipy).
# First run on Colab takes ~1-2 min; safe to re-run (a no-op once installed).
%pip install -q "iontrap-dynamics[plot] @ git+https://github.com/uwarring82/iontrap-dynamics.git@main"

# Tutorial 19 — Squeezing a trapped ion by quenching its trap frequency


**Goal.** Generate motional squeezing by varying an ion's trap frequency `ω(t)`
in time — no laser, just a fast change of the confinement. By the end you will
have built the time-dependent-frequency squeezing Hamiltonian, read the squeezing
back from the phase-space **covariance matrix**, seen the **sudden vs adiabatic**
crossover, watched the Wigner ellipse squeeze, and grown squeezing linearly by
**parametric modulation**. This reproduces the single-ion physics of Wittemer et
al., *Phil. Trans. R. Soc. A* **378**, 20190230 (2020).

**Reference implementation.** `tools/run_benchmark_nonadiabatic_squeezing.py`,
with the committed plot under
[`benchmarks/data/nonadiabatic_squeezing/`](https://github.com/uwarring82/iontrap-dynamics/tree/main/benchmarks/data/nonadiabatic_squeezing).

**Expected time.** ~14 min reading; ~3 s runtime.

**Level.** `advanced` — a specialised or research-grade surface; do the core first.

**Prerequisites.** [Tutorial 9](https://uwarring82.github.io/iontrap-dynamics/tutorials/09_squeezed_coherent_prep/) (single-mode
squeezed-state factory) and [Tutorial 6](https://uwarring82.github.io/iontrap-dynamics/tutorials/06_fock_truncation/) (Fock-truncation
diagnosis). CONVENTIONS.md **§26** fixes the squeezing generator, the
vacuum-variance-1 quadrature normalisation, and the Wigner scaling used here.

---

## The scenario

A harmonic oscillator whose frequency `ω(t)` is changed in time is one of the
oldest problems in quantum mechanics — and the workhorse of *analogue-gravity*
experiments. In the **fixed** operator basis of the initial frequency `ω(0)`, the
evolution is (CONVENTIONS §26.1, after Silveri 2015)

```
H(t)/ℏ = ω(t) (â†â + ½) − (i/4)(d ln ω/dt)(â†² − â²).
```

The second term — switched on **only while ω is changing** — is a squeezing
generator. Change `ω` slowly (adiabatically) and the state just follows the
instantaneous ground state; change it fast (non-adiabatically) and the state is
left **squeezed**: pairs of phonons are torn out of the vacuum. We drive this
with a [`FrequencyWaveform`](https://github.com/uwarring82/iontrap-dynamics/blob/main/src/iontrap_dynamics/waveforms.py)
that carries both `ω(t)` and its analytic log-derivative.

## Step 1 — Build the ω(t) squeezing Hamiltonian and evolve a fast ramp

We use a smooth `tanh` ramp from `ω_i` down to `ω_f = ½ ω_i`. Its total log-swing
is fixed at `ln(ω_f/ω_i)` regardless of width, so narrowing the ramp takes us to
the **sudden** limit, where the generated squeezing is exactly
`r = ½|ln(ω_f/ω_i)|`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import qutip

from iontrap_dynamics import gaussian, phase_space, waveforms
from iontrap_dynamics.hamiltonians import nonadiabatic_squeezing_hamiltonian
from iontrap_dynamics.hilbert import HilbertSpace
from iontrap_dynamics.modes import ModeConfig
from iontrap_dynamics.results import StorageMode
from iontrap_dynamics.sequences import solve
from iontrap_dynamics.species import mg25_plus
from iontrap_dynamics.system import IonSystem

BLUE, RED, GREEN, GREY = "#1f77b4", "#d62728", "#2ca02c", "#444444"
TWOPI = 2.0 * np.pi


def single_mode(fock, freq_hz=2.0e6):
    mode = ModeConfig(
        label="m",
        frequency_rad_s=TWOPI * freq_hz,
        eigenvector_per_ion=np.array([[0.0, 0.0, 1.0]]),
    )
    system = IonSystem(species_per_ion=(mg25_plus(),), modes=(mode,))
    return HilbertSpace(system=system, fock_truncations={"m": fock})


def evolve(hilbert, wave, tmax, n_times):
    fock = hilbert.fock_truncations["m"]
    psi0 = qutip.tensor(qutip.basis(2, 0), qutip.basis(fock, 0))
    hamiltonian = nonadiabatic_squeezing_hamiltonian(hilbert, "m", wave, validate_at=(0.0, tmax))
    return solve(
        hilbert=hilbert,
        hamiltonian=hamiltonian,
        initial_state=psi0,
        times=np.linspace(0.0, tmax, n_times),
        storage_mode=StorageMode.EAGER,
    )


w_i, w_f = 2.0e6, 1.0e6
hilbert = single_mode(fock=40, freq_hz=w_i)
period = 1.0 / w_i
width = 0.01 * period  # narrow → sudden
tmax = 55.0 * width
n_times = 1500
ramp = waveforms.smooth_ramp(
    omega_i=TWOPI * w_i, omega_f=TWOPI * w_f, center_s=25.0 * width, width_s=width
)
result = evolve(hilbert, ramp, tmax, n_times)
mode_state = gaussian.reduced_single_mode(result.states[-1], hilbert, "m")

r_sudden = gaussian.squeezing_parameter(gaussian.covariance_matrix(mode_state)[0])
r_oracle = 0.5 * abs(np.log(w_f / w_i))
print(f"sudden ramp: r = {r_sudden:.4f}   oracle ½|ln(ω_f/ω_i)| = {r_oracle:.4f}")
assert abs(r_sudden - r_oracle) / r_oracle < 0.05

The squeezing appears **exactly while `ω(t) is changing`**, then stays put. Reading
the covariance along the trajectory (the eigenvalue-ratio `r` is rotation-invariant,
so it plateaus after the quench) tells the whole story in one figure:

In [ ]:
t_grid = np.linspace(0.0, tmax, n_times)
sample = np.arange(0, n_times, 20)
omega_profile = np.array([ramp.omega(t) for t in t_grid[sample]]) / (TWOPI * w_i)
r_of_t = np.array([
    gaussian.squeezing_parameter(
        gaussian.covariance_matrix(gaussian.reduced_single_mode(result.states[k], hilbert, "m"))[0]
    )
    for k in sample
])

fig, ax = plt.subplots(figsize=(6.5, 4.0))
t_us = t_grid[sample] * 1e6
ax.plot(t_us, r_of_t, color=BLUE, lw=2, label="squeezing r(t)")
ax.axhline(r_oracle, color=RED, ls="--", label="sudden oracle")
ax.set_xlabel("time [µs]")
ax.set_ylabel("squeezing r", color=BLUE)
ax.set_title("squeezing appears exactly when the trap is quenched")
ax.legend(loc="center right")
ax2 = ax.twinx()
ax2.plot(t_us, omega_profile, color=GREEN, ls="--")
ax2.set_ylabel("trap frequency ω(t)/ω_i", color=GREEN)
fig.tight_layout()

# r is rotation-invariant, so its final value is the sudden oracle.
assert abs(r_of_t[-1] - r_oracle) / r_oracle < 0.05

## Step 2 — Read the squeezing back from the covariance matrix

The readout lives in [`iontrap_dynamics.gaussian`](https://github.com/uwarring82/iontrap-dynamics/blob/main/src/iontrap_dynamics/gaussian.py).
From the reduced mode state it builds the 2×2 covariance `V` (quadratures
`x̂ = â + â†`, `p̂ = i(â† − â)`, vacuum variance 1) and reports the squeezing
`r = ¼ ln(λ_max/λ_min)` (the **eigenvalue ratio**, not `tr V`), the symplectic
eigenvalue `ν = √(det V)` (purity: `ν = 1` for a pure state), and `n̄_sq = sinh²r`.

In [ ]:
readout = phase_space.phase_space_readout(mode_state)
print(f"r = {readout.squeezing_parameter:.4f}   ν = {readout.symplectic_eigenvalue:.4f}"
      f"   n̄_sq = {readout.mean_squeezed_occupation:.4f}   |α| = {abs(readout.coherent_amplitude):.2e}")

# The state stayed pure (ν = 1) and displacement-free (the centred generator
# preserves parity: ⟨â⟩ = 0 from vacuum).
assert abs(readout.symplectic_eigenvalue - 1.0) < 1e-3
assert abs(readout.coherent_amplitude) < 1e-6
assert abs(readout.mean_squeezed_occupation - np.sinh(r_oracle) ** 2) / np.sinh(r_oracle) ** 2 < 0.1

## Step 3 — Sudden kick vs cyclic adiabatic return

Step 1 gave the **sudden** one-way squeeze kick: a narrow ramp from `ω_i` to
`ω_f` gives `r = ½|ln(ω_f/ω_i)|`. The clean **adiabatic** oracle in §26 is
cyclic: ramp down slowly and then back up to the original frequency. In that
case the state returns to the original oscillator and the residual squeezing
goes to zero (`r → 0`). Do not use a one-way frequency change as the `r → 0`
regression; the convention gate is the cyclic waveform.

In [ ]:
def cyclic_down_up(width_s):
    """Smoothly ramp ω_i → ω_f → ω_i with analytic d ln ω/dt."""
    first_center = 5.0 * width_s
    second_center = 15.0 * width_s
    ln_half_swing = 0.5 * np.log(w_f / w_i)

    def omega(t):
        log_swing = ln_half_swing * (
            np.tanh((t - first_center) / width_s) - np.tanh((t - second_center) / width_s)
        )
        return TWOPI * w_i * np.exp(log_swing)

    def d_ln_omega_dt(t):
        s1 = 1.0 / np.cosh((t - first_center) / width_s) ** 2
        s2 = 1.0 / np.cosh((t - second_center) / width_s) ** 2
        return ln_half_swing * (s1 - s2) / width_s

    return waveforms.FrequencyWaveform(omega=omega, d_ln_omega_dt=d_ln_omega_dt)


widths = np.array([0.1, 0.3, 1.0, 3.0]) * period
r_cyclic = []
for w in widths:
    hil_c = single_mode(50, w_i)
    state = gaussian.reduced_single_mode(evolve(hil_c, cyclic_down_up(w), 20.0 * w, 2500).states[-1], hil_c, "m")
    r_cyclic.append(gaussian.squeezing_parameter(gaussian.covariance_matrix(state)[0]))
r_cyclic = np.array(r_cyclic)
print("width/T_i :", widths / period)
print("cyclic r  :", np.round(r_cyclic, 5))

# The one-way sudden kick hits the analytic oracle; the cyclic ramp's residual
# squeezing vanishes as the down/up waveform becomes adiabatic.
assert abs(r_sudden - r_oracle) / r_oracle < 0.05
assert np.all(np.diff(r_cyclic) < 0.0)
assert r_cyclic[-1] < 1e-2

fig, ax = plt.subplots(figsize=(6.0, 4.0))
ax.semilogx(widths / period, r_cyclic, "o-", color=BLUE, label="cyclic down/up residual")
ax.axhline(r_oracle, color=RED, ls="--", label="one-way sudden kick")
ax.axhline(0.0, color=GREY, ls=":", label="adiabatic limit")
ax.set_xlabel("ramp width / trap period")
ax.set_ylabel("squeezing r")
ax.set_title("cyclic residual squeezing vanishes adiabatically")
ax.legend()
fig.tight_layout()

## Step 4 — See it in phase space: the Wigner ellipse

The [`phase_space.wigner`](https://github.com/uwarring82/iontrap-dynamics/blob/main/src/iontrap_dynamics/phase_space.py)
wrapper pins QuTiP's scaling to the §26.2 vacuum-variance-1 convention (`g = 1`),
so the Wigner ellipse's principal widths are exactly the covariance eigenvalues
`e^{∓2r}`. The vacuum is a unit circle; squeezing flattens it.

In [ ]:
grid = np.linspace(-4.0, 4.0, 201)
w_vac = phase_space.wigner(qutip.basis(40, 0), grid)
w_sqz = phase_space.wigner(mode_state, grid)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(9.0, 4.2))
for ax, w, title in ((ax0, w_vac, "vacuum"), (ax1, w_sqz, f"squeezed (r = {r_sudden:.2f})")):
    ax.contourf(grid, grid, w, levels=30, cmap="viridis")
    ax.set_aspect("equal")
    ax.set_xlabel(r"$\hat x$")
    ax.set_ylabel(r"$\hat p$")
    ax.set_title(title)
fig.tight_layout()

# The squeezed ellipse's narrow axis is e^{-2r} of the vacuum's unit variance.
cov, _ = gaussian.covariance_matrix(mode_state)
narrow_axis = float(np.min(np.linalg.eigvalsh(cov)))
assert abs(narrow_axis - np.exp(-2.0 * r_sudden)) < 0.02

## Step 5 — Grow squeezing continuously: parametric modulation

Instead of one fast quench, **modulate** `ω(t)` sinusoidally at twice the trap
frequency, `ω_mod = 2 ω_ini`. This is degenerate parametric amplification: the
squeezing grows **linearly** with the modulation duration, `r = ½ δω · T_mod`, so
`n̄_sq = sinh²(2π g T_mod)` with coupling `g = δω/(4π)`. It is intrinsically
displacement-free.

In [ ]:
w_ini = 2.8e6
dw_mod = TWOPI * 8.0e3
hil_p = single_mode(fock=40, freq_hz=w_ini)
mod_period = TWOPI / (2.0 * TWOPI * w_ini)
t_list = np.array([40, 90, 140]) * mod_period
r_param = []
for tmax in t_list:
    wave = waveforms.sinusoidal_modulation(
        omega_ini=TWOPI * w_ini, mod_amplitude=dw_mod, mod_frequency=2.0 * TWOPI * w_ini
    )
    st = gaussian.reduced_single_mode(evolve(hil_p, wave, float(tmax), 2200).states[-1], hil_p, "m")
    r_param.append(gaussian.squeezing_parameter(gaussian.covariance_matrix(st)[0]))
r_param = np.array(r_param)
r_pred = 0.5 * dw_mod * t_list
print("T_mod [µs]:", np.round(t_list * 1e6, 2))
print("r sim     :", np.round(r_param, 4))
print("r = ½δω·T :", np.round(r_pred, 4))

# Linear growth in the modulation duration.
assert np.allclose(r_param, r_pred, rtol=0.03)

# The pairs pile up: n̄_sq = sinh²r grows as sinh²(½δω·T_mod).
t_fine = np.linspace(0.0, t_list[-1], 100)
fig, ax = plt.subplots(figsize=(6.5, 4.0))
ax.plot(t_fine * 1e6, np.sinh(0.5 * dw_mod * t_fine) ** 2, color=BLUE,
        label=r"$\sinh^2(\frac{1}{2}\delta\omega\,T_{mod})$")
ax.plot(t_list * 1e6, np.sinh(r_param) ** 2, "o", color=RED, ms=8, label="simulation")
ax.set_xlabel("modulation duration [µs]")
ax.set_ylabel(r"mean squeezed phonons $\bar n_{sq}$")
ax.set_title("parametric amplification: squeezing grows with duration")
ax.legend()
fig.tight_layout()

## What you built

You generated squeezing with nothing but a time-dependent trap frequency, read it
back from the covariance matrix (the eigenvalue-ratio `r`, the purity `ν`, the
occupation `n̄_sq`), checked the sudden kick and cyclic adiabatic limits, visualised
the squeezed Wigner ellipse on the vacuum-variance-1 grid, and grew squeezing
linearly by parametric modulation. [Tutorial 20](https://uwarring82.github.io/iontrap-dynamics/tutorials/20_phonon_pair_creation/) reads the same
states out as a **phonon-number distribution** and shows the even-only
phonon-**pair** signature.